# Querying EFD Data in Jupyter: Examples

In this workbook we are going to show some examples of how to query EFD (Engineering and Facility Database) data from a Jupyter workbook.


In [ ]:
import asyncio
import glob
import os
import sys
import time

import numpy as np
import pandas as pd
from astropy.time import Time, TimeDelta

from lsst.sitcom.vandv.logger import create_logger
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState

## Basic information

### Create a Client
First, we need to create a client to retrieve datasets from the EFD database.

In [ ]:
# Create an EFD client instance
client = makeEfdClient()

### Available Topics in EFD and how to access them
If you need to see the different topics available in the EFD, you can list them with the following command.

In [ ]:
# make a list of all different topics that you can query in the EFD
await client.get_topics()

Let's take an example where you want to analyse how the position of the **azimuth** behaved on a particular night of observation. The first thing you need to know is how to get to that information. If you don't know **the topic** you can list them all as we have done before but maybe you know a keyword related to the topic you are interested in, you can perform a more precise search like in this example:

In [ ]:
# make a list of all topics in the EFD related to MTMount
topics = await client.get_topics()
for topic in topics:
    if 'MTMount' in topic:
        print(topic)

After analysing the list of topics you have seen that the one you are interested in is **lsst.sal.MTMount.azimuth** (in our example). 

We can explore the **different fields** available for lsst.sal.MTMount.azimuth as follows

In [ ]:
# get all fields related to the MT Mount azimuth
await client.get_fields('lsst.sal.MTMount.azimuth')

Now we have all the information to access the azimuthal position for the day of our want by indicating the day and timestamp data within a specific time range, use the following command.

In [ ]:
# Get all azimuth position and timestamp data within a particular time range
start = Time("2023-03-10T03:00:00Z", scale="utc")
end = Time("2023-03-10T03:30:00Z", scale="utc")

az = await client.select_time_series(
    "lsst.sal.MTMount.azimuth", ["actualPosition", "timestamp"], start, end
)

We can  visualise this information

In [ ]:
type(az)

In [ ]:
len(az)

In [ ]:
az.columns

In [ ]:
# plot the actual position
az['actualPosition'].plot()

## Querying EFD Data with getEfdData

We can also access information from the EDF (Engineering and Facility Database) using `getEfdData` and `TMAEventMaker`. `getEfdData` allows to analyse an particular event or a set or events of a observation night. 

In [ ]:
# Create an instance of TMAEventMaker and initialize en EFD client
event_maker = TMAEventMaker()
client = makeEfdClient()

In the `getEfdData` function it is necessary to indicate in addition to the topic and the filds, the **event** we want. We can how many events are associated with an observation night as follows

In [ ]:
# Define the observation day
day_obs = 20231212

# Retrieve events for the specified observation day
events = event_maker.getEvents(day_obs)

# Create a logger instance for the events
log = create_logger("Eventos_20231212")
log.info(f"Found {len(events)} events for day {day_obs}")

If you know the event number:

In [ ]:
# Define the event number and create an event maker instance
event_num = 300

event_maker = TMAEventMaker()

# Retrieve the event for the specified day and slew ID
evt = event_maker.getEvent(day_obs, event_num)

Now we can make the call to `getEfdData` as follows:

In [ ]:
df = getEfdData(client, 'lsst.sal.MTMount.azimuth', columns=['actualPosition', 'timestamp'], event=evt)

In [ ]:
df['actualPosition'].plot()

You may not know the event number, so let's see how to find it. Let's assume you only know what the observation night is and when the event starts and ends.

In [ ]:
# Define the start and end date-time of the event
begin="2023-07-28T02:17:15"
end="2023-07-28T02:17:55"

With this information we will search for the event number.

In [ ]:
# Convert begin and end times to astropy Time objects in ISO 8601 format
time_begin = Time(begin, format="isot", scale="utc")
time_end = Time(end, format="isot", scale="utc")

# Calculate the midpoint time
time_half = time_begin + (time_end - time_begin) * 0.5

# Find the event based on the midpoint time
event = event_maker.findEvent(time_half)

# Print event details
print(
    f"Event from {begin=} to {end=} "
    f"and has sequence number {event.seqNum} "
    f"and observation day {event.dayObs}"
)

We now have all the information we need to make the call to `getEfdData`

In [ ]:
df = getEfdData(
            client,
            topic= 'lsst.sal.MTMount.azimuth',
            columns=['actualPosition', 'timestamp'],
            event=event
        )

In [ ]:
df

## Querying the EFD with InfluxQL


We can also access the data using the EFD client together with InfluxQL to query the EFD database.  You can follow this notebook for more details: https://github.com/lsst-sqre/sasquatch/blob/main/docs/user-guide/notebooks/UsingInfluxQL.ipynb

You need to import the library

In [ ]:
from lsst_efd_client import EfdClient

In [ ]:
# Create an EFD client instance
client = EfdClient('usdf_efd')

We can list the topis of the client

In [ ]:
topics = await client.get_topics()
[topic for topic in topics if "lsst.sal.MTMount" in topic]

You can visualise the structure and organisation of the data in the EFD client.

In [ ]:
schema = await client.get_schema("lsst.sal.MTMount.azimuth")
schema

In [ ]:
begin = "2023-07-28T02:17:15"
end = "2023-07-28T02:17:55"

# Convert to Time objects
time_begin = Time(begin, format="isot", scale="utc")
time_end = Time(end, format="isot", scale="utc")

# Convert to strings in the correct format for InfluxDB (ISO 8601 with "Z")
time_begin_str = time_begin.to_value("isot")  # "2023-07-28T02:17:15"
time_end_str = time_end.to_value("isot")      # "2023-07-28T02:17:55"

# Add "Z" at the end (to indicate UTC in InfluxDB)
time_begin_str += "Z"
time_end_str += "Z"

In [ ]:
# Query
query = f'''SELECT "actualPosition" FROM "lsst.sal.MTMount.azimuth" 
            WHERE time > '{time_begin_str}' AND time < '{time_end_str}' '''

df = await client.influx_client.query(query)


In [ ]:
df